In [1]:
import nflreadpy as nfl
import pathlib
import polars as pl

In [2]:
# Player data

# seasons to retrieve data for
seasons_to_retrieve = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# player positions
all_positions = ['P', 'TE', 'OT', 'QB', 'FB', 'S', 'C', 'CB', 'DL', 'WR', 'MLB', 'OLB', 'DE', None, 'DB', 'LS', 'NT',
                 'SAF', 'G', 'DT', 'ILB', 'FS', 'K', 'RB', 'LB']

# player positions needed for fantasy scoring metrics
fantasy_relevant_positions = ['TE', 'QB', 'WR', 'K', 'RB']

# fantasy scoring fields (with a few extra for info)
fantasy_cols = [
    # Player identification
    'player_id', 'player_display_name', 'position', 'recent_team', 'season',

    # Passing (QB)
    'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'passing_2pt_conversions',

    # Rushing (RB/QB)
    'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost', 'rushing_2pt_conversions',

    # Receiving (WR/RB/TE) - PPR CRITICAL
    'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles_lost',
    'receiving_2pt_conversions',

    # Kicking
    'fg_made', 'fg_missed', 'pat_made', 'fg_made_0_19', 'fg_made_20_29', 'fg_made_30_39', 'fg_made_40_49',
    'fg_made_50_59', 'fg_made_60_',

    # ALREADY CALCULATED (use these directly)
    'fantasy_points', 'fantasy_points_ppr'
]



In [3]:
# retrieval choices
positions_to_retrieve = fantasy_relevant_positions
cols_to_retrieve = fantasy_cols

# create a query to get the relevant data
get_relevant_player_data = (nfl.load_player_stats(
    seasons=seasons_to_retrieve,
    summary_level="reg"
)[cols_to_retrieve].lazy()
                            .filter(pl.col('position').is_in(positions_to_retrieve))
                            .sort(['season', 'position'])
                            )

player_data = get_relevant_player_data.collect()

path = pathlib.Path.cwd() / "player_data_2016_2025.csv"

print(player_data.write_csv(str(path), include_header=True))

None
